# Office Rules Assistant — Mini RAG Project

**Goal:** Ask questions about a fictional workplace handbook and retrieve the most relevant rule.

This notebook is designed for **Google Colab**. It uses only Python's standard library, so no API key or package installation is required. It demonstrates a simple Retrieval-Augmented Generation (RAG) workflow: store knowledge → split into chunks → retrieve relevant chunks → show the answer from the source text.

> The office policies below are fictional examples for this project, not legal or HR advice.

In [1]:
# 1. Fictional office rules: distinctive and practical policies
office_rules = [
    "Focus-hour rule: From 10:00 AM to 12:00 PM, employees should avoid non-urgent calls and meetings so teammates can do concentrated work.",
    "Meeting reset rule: The meeting organiser must share a three-bullet agenda at least 2 hours before a meeting; meetings without an agenda should be rescheduled unless urgent.",
    "Quiet-zone rule: The two desks beside the library corner are silent workspaces; calls must be taken in a phone booth or outside that zone.",
    "Green commute rule: Employees who cycle, walk, or use public transport at least three days a week may join the monthly green-commute recognition draw.",
    "Desk reset rule: Before leaving, each employee should clear food waste, secure confidential papers, and leave shared desks ready for the next person.",
    "Meeting-free lunch rule: No internal meetings should be scheduled between 1:00 PM and 1:30 PM, except for urgent incidents or client time-zone needs.",
    "Device charging rule: Personal devices may be charged only at designated charging stations; cables must not cross walkways.",
    "Idea-lab rule: On the first Wednesday of each month, teams may use 30 minutes to pitch one small process improvement, even if it is only a rough idea.",
    "Shared-snack rule: Label food placed in the shared fridge with your name and the date; unlabelled items older than three days may be removed during Friday cleaning.",
    "Printer-saving rule: Drafts and internal reading copies should be printed double-sided and in grayscale when practical.",
    "Work-from-home handover rule: Remote employees should post a short status update, next step, and blocker (if any) in the team channel before signing off.",
    "Lost-item rule: Found keys, access cards, or devices must be handed to reception; do not post photos of access cards in public chat channels.",
    "Room-booking rule: Release a meeting-room booking if the team no longer needs it; rooms unclaimed 10 minutes after the start time may be made available to others.",
    "Wellbeing pause rule: Employees may take a brief screen break between tasks; teams should avoid treating a short wellbeing break as a sign of being unavailable for the whole day.",
    "Confidentiality rule: Customer records and internal documents must be stored only in company-approved locations, not in personal email or public file-sharing links."
]

print(f"Loaded {len(office_rules)} office rules.")
for i, rule in enumerate(office_rules, start=1):
    print(f"{i:02d}. {rule}")

Loaded 15 office rules.
01. Focus-hour rule: From 10:00 AM to 12:00 PM, employees should avoid non-urgent calls and meetings so teammates can do concentrated work.
02. Meeting reset rule: The meeting organiser must share a three-bullet agenda at least 2 hours before a meeting; meetings without an agenda should be rescheduled unless urgent.
03. Quiet-zone rule: The two desks beside the library corner are silent workspaces; calls must be taken in a phone booth or outside that zone.
04. Green commute rule: Employees who cycle, walk, or use public transport at least three days a week may join the monthly green-commute recognition draw.
05. Desk reset rule: Before leaving, each employee should clear food waste, secure confidential papers, and leave shared desks ready for the next person.
06. Meeting-free lunch rule: No internal meetings should be scheduled between 1:00 PM and 1:30 PM, except for urgent incidents or client time-zone needs.
07. Device charging rule: Personal devices may be ch

## 2. Split the rules into searchable chunks

For a small, self-contained demo, each rule is already a useful chunk. This avoids breaking a policy across multiple chunks and keeps the answer traceable to its source.

In [2]:
# Each complete rule is one document/chunk.
documents = [{"id": i + 1, "text": rule} for i, rule in enumerate(office_rules)]

# Small standard-library tokenizer; it normalises case and ignores punctuation.
import re

def tokenize(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))

# Common words add little value when matching a question to a rule.
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "can", "do", "does", "for",
    "how", "i", "in", "is", "it", "me", "my", "of", "on", "or", "should",
    "the", "to", "we", "what", "when", "where", "which", "who", "why", "with"
}

def useful_tokens(text):
    return tokenize(text) - STOP_WORDS

print("Prepared", len(documents), "searchable chunks.")

Prepared 15 searchable chunks.


## 3. Retrieval function

The function scores each rule by how many meaningful words overlap with the question. It returns the best-matching rule and its source ID. If there is no useful overlap, it says that no matching rule was found rather than inventing a policy.

In [3]:
def retrieve(question, top_k=2):
    question_terms = useful_tokens(question)
    if not question_terms:
        return []

    scored = []
    for doc in documents:
        rule_terms = useful_tokens(doc["text"])
        overlap = question_terms & rule_terms
        # Jaccard-like score keeps long rules from automatically winning.
        union = question_terms | rule_terms
        score = len(overlap) / len(union) if union else 0.0
        if overlap:
            scored.append((score, len(overlap), doc))

    scored.sort(key=lambda item: (item[0], item[1]), reverse=True)
    return [(score, doc) for score, _, doc in scored[:top_k]]


def ask(question):
    print("Q:", question)
    results = retrieve(question)
    if not results:
        print("A: No matching office rule found. Please check the official handbook or ask HR.")
    else:
        best_score, best_doc = results[0]
        print(f"A: {best_doc['text']}")
        print(f"Source rule: {best_doc['id']} | Match score: {best_score:.3f}")
        if len(results) > 1:
            print("Other possible match:", results[1][1]["text"])
    print()

## 4. Run test questions

Run this cell to check that the assistant can retrieve different policies. You can edit or add questions.

In [4]:
test_questions = [
    "When should I avoid scheduling meetings?",
    "What should an agenda include and when must it be shared?",
    "Where can I take a phone call?",
    "What do remote workers post before signing off?",
    "How should confidential customer files be stored?",
    "What happens to a room booking if nobody arrives?",
    "Can I leave food in the shared fridge without a label?",
    "What is the rule for printing drafts?",
    "What should I do if I find an access card?",
    "Does the office have a rule about cycling to work?",
    "What is the office dress code?"  # Not covered by the rules: should not invent an answer.
]

for question in test_questions:
    ask(question)

Q: When should I avoid scheduling meetings?
A: Focus-hour rule: From 10:00 AM to 12:00 PM, employees should avoid non-urgent calls and meetings so teammates can do concentrated work.
Source rule: 1 | Match score: 0.100
Other possible match: Meeting reset rule: The meeting organiser must share a three-bullet agenda at least 2 hours before a meeting; meetings without an agenda should be rescheduled unless urgent.

Q: What should an agenda include and when must it be shared?
A: Meeting reset rule: The meeting organiser must share a three-bullet agenda at least 2 hours before a meeting; meetings without an agenda should be rescheduled unless urgent.
Source rule: 2 | Match score: 0.100
Other possible match: Device charging rule: Personal devices may be charged only at designated charging stations; cables must not cross walkways.

Q: Where can I take a phone call?
A: Quiet-zone rule: The two desks beside the library corner are silent workspaces; calls must be taken in a phone booth or outsid

## 5. Try your own question

Edit `my_question` and run the cell. The answer is retrieved from the rules above; this demo does not call an external language model.

In [5]:
my_question = "What should I do before leaving my desk?"
ask(my_question)

Q: What should I do before leaving my desk?
A: Desk reset rule: Before leaving, each employee should clear food waste, secure confidential papers, and leave shared desks ready for the next person.
Source rule: 5 | Match score: 0.158
Other possible match: Meeting reset rule: The meeting organiser must share a three-bullet agenda at least 2 hours before a meeting; meetings without an agenda should be rescheduled unless urgent.



## How this mini RAG demo works

1. **Knowledge base:** the `office_rules` list stores the handbook facts.
2. **Chunking:** each rule is kept as a separate chunk.
3. **Retrieval:** `retrieve()` ranks rules using token overlap.
4. **Grounded response:** `ask()` displays the best source rule and its ID.

**Limitation:** this is a lightweight keyword-retrieval demonstration, not a production semantic-search system. Similar wording works best. For a more advanced version, you could replace the retrieval function with embeddings/vector search and connect a language model.